# Code RL with Harbor hello-world + Modal sandboxes

This tutorial trains a model on the
[hello-world](https://hub.harborframework.com/tasks/habor/hello-world/latest)
task from Harbor Hub, scoring solutions by executing them in Modal sandboxes.

Workflow:
1. Pull the hello-world task from Harbor Hub via `HarborDataset`.
2. Score model outputs by executing code in Modal sandboxes,
   piping test inputs via stdin and comparing stdout.
3. Use that scorer for both offline eval and as SLIME `custom_rm_function`.
4. Train and compare base vs. trained behavior.

The key pattern: **correctness drives reward**.
Reward = fraction of test cases whose output matches expected.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main harbor

In [ ]:
import json
import re

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HarborDataset,
    ModelDeployment,
    Qwen3_4B,
    SlimeRecipe,
    TrainConfig,
    list_checkpoints,
)

## Load hello-world from Harbor Hub

`HarborDataset` accepts a `dataset_name` to pull tasks from
[Harbor Hub](https://hub.harborframework.com). Each task has:
- `instruction.md` — the problem statement (prompt)
- `task.toml` — metadata (difficulty, category)
- `tests/` — verification tests (format varies by task)

The hello-world task uses pytest-based verification rather than
`*.in`/`*.out` file pairs, so we define stdin/stdout test cases
inline and pass them directly to the sandbox scorer.

A single dataset instance handles both training and eval —
`prepare()` writes train and eval splits to the volume,
while `load()` returns all tasks for offline evaluation.

In [ ]:
HELLO_WORLD_TESTS = [{"input": "", "expected_output": "Hello, world!\n"}]

dataset = HarborDataset(
    dataset_name="harbor/hello-world",
    label_metadata_path="task.toml",
    train_repeats=20,
    always_prepare=True, # For the purpose of this tutorial, we want to prepare the dataset every time we run it, in case there is stale data from a previous run.
    system_prompt=(
        "You are an expert Python programmer. "
        "Solve the given problem by writing a complete Python program. "
        "Your program must print the answer to stdout using print(). "
        "Do not create or write any files. "
        "Put your solution in a ```python code fence."
    ),
)

Let's take a quick look at part of the dataset as a pandas DataFrame.
Each row includes the task prompt plus the parsed Harbor label metadata.

In [ ]:
df = dataset.to_pandas()
print(len(df))
df.head(5)

## Sandbox-backed scorer

We execute candidate code in a Modal sandbox with test inputs piped via
`sys.stdin`, then compare stdout against expected output. All test cases
run in a single sandbox per sample for efficiency.

In [ ]:
_CODE_FENCE_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)

def extract_python_code(text: str) -> str:
    normalized = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    if "<|im_start|>assistant" in normalized:
        normalized = normalized.rsplit("<|im_start|>assistant", 1)[-1]
    if "</think>" in normalized:
        normalized = normalized.split("</think>", 1)[-1]
    normalized = normalized.replace("<think>", "").replace("<|im_end|>", "").strip()
    if match := _CODE_FENCE_RE.search(normalized):
        return match.group(1).strip()
    return normalized

def score_usaco_with_sandbox(
    response: str,
    *,
    test_cases: list[dict],
    timeout_sec: int = 60,
) -> tuple[float, dict]:
    import modal as _modal

    code = extract_python_code(response)
    if not test_cases:
        return 0.0, {"passed": 0, "total": 0}

    script = "\n".join([
        "import sys, io, json",
        f"candidate = {json.dumps(code)}",
        f"tests = {json.dumps(test_cases)}",
        "results = []",
        "for tc in tests:",
        "    sys.stdin = io.StringIO(tc['input'])",
        "    buf = io.StringIO()",
        "    old = sys.stdout",
        "    sys.stdout = buf",
        "    try:",
        "        exec(candidate, {}, {})",
        "        sys.stdout = old",
        "        results.append({'output': buf.getvalue(), 'ok': True})",
        "    except Exception as e:",
        "        sys.stdout = old",
        "        results.append({'output': '', 'ok': False})",
        "print(json.dumps(results))",
    ]) + "\n"

    try:
        app = _modal.App.lookup("training-gym-sandbox-rm", create_if_missing=True)
        sandbox = _modal.Sandbox.create(
            "python", "-c", script,
            app=app,
            image=_modal.Image.debian_slim(python_version="3.11"),
            timeout=timeout_sec,
            cpu=1.0,
            memory=1024,
        )
        stdout = sandbox.stdout.read()
        sandbox.stderr.read()
        sandbox.wait()
    except Exception:
        return 0.0, {"passed": 0, "total": len(test_cases)}

    try:
        results = json.loads(stdout)
    except (json.JSONDecodeError, ValueError):
        return 0.0, {"passed": 0, "total": len(test_cases)}

    passed = sum(
        1 for r, tc in zip(results, test_cases)
        if r.get("ok") and r.get("output", "").strip() == tc["expected_output"].strip()
    )
    return passed / len(test_cases), {"passed": passed, "total": len(test_cases)}

async def usaco_rm(args, sample, **kwargs) -> float:
    import asyncio

    reward, meta = await asyncio.to_thread(
        score_usaco_with_sandbox, sample.response, test_cases=HELLO_WORLD_TESTS,
    )
    sample.metadata = {**(getattr(sample, "metadata", None) or {}), "usaco": meta}
    return float(reward)

## Serve and evaluate the base model

In [ ]:
base_model = Qwen3_4B()
base_deployment: ModelDeployment = DeploymentConfig(model=base_model).serve()
print(f"Base model URL: {base_deployment.url}")

def eval_fn(deployment: ModelDeployment, example: dict) -> EvalRowResult:
    prompt = example.get("instruction", "")
    response = deployment.generate(
        prompt,
        ensure_ready=False,
        messages=[
            {"role": "system", "content": dataset.system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    score, metadata = score_usaco_with_sandbox(response, test_cases=HELLO_WORLD_TESTS)
    return EvalRowResult(score=score, response=response, metadata=metadata)

eval_config = EvalConfig(
    dataset=dataset,
    eval_fn=eval_fn,
)
print("Running base eval...")
base_eval = eval_config.evaluate(base_deployment, debug=True)
print(f"Base mean reward: {base_eval.mean:.4f}")

## Train with SLIME and sandbox reward

In [ ]:
training_run = TrainConfig(
    model=Qwen3_4B(),
    dataset=dataset,
    recipe=SlimeRecipe(
        custom_rm_function=usaco_rm,

        gpu_type="H100",
        colocate=True,
        tensor_model_parallel_size=1,
        sequence_parallel=False,
        rollout_num_gpus_per_engine=1,

        num_rollout=10,
        rollout_batch_size=8,
        n_samples_per_prompt=8,
        rollout_max_response_len=2048,
        rollout_temperature=0.9,

        global_batch_size=8,
        eval_max_response_len=2048,
        n_samples_per_eval_prompt=8,
        max_tokens_per_gpu=4096,
        save_interval=10,
        image_overlay=lambda image: image.run_commands(
            "uv pip install --system modal>=1.2.0",
        ),
    ),
)
print("Starting training...")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")

## Evaluate the trained checkpoint

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
trained_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=checkpoint,
    app_name="qwen3-4b-hello-world-serve",
    served_model_name="qwen3-4b-hello-world",
).serve()
print(f"Trained model URL: {trained_deployment.url}")

trained_eval = eval_config.evaluate(trained_deployment, debug=True)
print(f"Trained mean reward: {trained_eval.mean:.4f}")
print(f"Base mean reward:    {base_eval.mean:.4f}")